In [53]:
import requests
import json
import re
import pandas as pd
import numpy as np

def get_uniprot(accession):
  '''
  define http_function to get the data from Uniprot API
  '''
  endpoint = "https://rest.uniprot.org/uniprotkb/accessions"
  http_function = requests.get
  http_args = {'params': {'accessions': accession}}
  return http_function(endpoint, **http_args)

def uniprot_parse_response(resp: dict):
  '''
  parse response from Uniprot and output
  organism, geneInfo, sequenceInfo, type

  do not forget to include error handling (e.g. for key errors)
  '''
  try:
    data = resp.json()

    entry = data['results'][0]
    return {
        'organism': entry.get('organism', {}).get('scientificName'),
        'geneInfo': entry.get('genes'),
        'sequenceInfo': entry.get('sequence'),
        'type': entry.get('entryType')
        }
  except Exception as e:
    return e

def get_ensembl(id):
  '''
  define http_function to get the data from Ensembl API
  '''
  endpoint = f"https://rest.ensembl.org/lookup/id/{id}"
  http_function = requests.get
  http_args = {"headers": {"Content-Type": "application/json"}}
  return http_function(endpoint, **http_args)

def ensembl_parse_response(resp: dict):
  '''
  parse Ensembl response and output
  object_type, assembly_name, species, db_type, biotype, display_name, id, description, canonical_transcript, source

  do not forget to include error handling (e.g. for key errors)
  '''
  try:
    data = resp.json()
    fields = [
        'object_type', 'species', 'assembly_name', 'biotype',
        'display_name', 'id', 'db_type', 'description', 'source',
        'canonical_transcript'
    ]
    return {field: data.get(field) for field in fields}
  except Exception as e:
    return e

def main(ids: list):
  '''
  parse Ensembl response and output
  object_type, assembly_name, species, db_type, biotype, display_name, id, description, canonical_transcript, source

  do not forget to include error handling (e.g. for key errors)
'''
  uniprot_pattern = r'^[A-Z][0-9][A-Z0-9]+'
  ensembl_pattern = r'^ENS[A-Z]*[0-9]+'

  output = {}

  for entry_id in ids:
      if re.match(uniprot_pattern, entry_id):
          res = get_uniprot(entry_id)
          if res.status_code == 200:
              output[entry_id] = uniprot_parse_response(res)
          else:
              output[entry_id] = {"error": f"HTTP {res.status_code}"}

      elif re.match(ensembl_pattern, entry_id):
          res = get_ensembl(entry_id)
          if res.status_code == 200:
              output[entry_id] = ensembl_parse_response(res)
          else:
              output[entry_id] = {"error": f"HTTP {res.status_code}"}
      else:
          output[entry_id] = {"error": "unknown database"}

  df = pd.DataFrame.from_dict(output, orient='index')
  return df

In [54]:
get_uniprot('P11473')


<Response [200]>

In [55]:
get_uniprot('helloworld')


<Response [400]>

In [56]:
get_uniprot('helloworld').json()


{'url': 'http://rest.uniprot.org/uniprotkb/accessions',
 'messages': ["Accession 'helloworld' has invalid format. It should be a valid UniProtKB accession with optional sequence range e.g. P12345[10-20]."]}

In [57]:
uniprot_parse_response(get_uniprot('P11473'))


{'organism': 'Homo sapiens',
 'geneInfo': [{'geneName': {'evidences': [{'evidenceCode': 'ECO:0000312',
      'source': 'HGNC',
      'id': 'HGNC:12679'}],
    'value': 'VDR'},
   'synonyms': [{'value': 'NR1I1'}]}],
 'sequenceInfo': {'value': 'MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNESFTMDDMSWTCGNQDYKYRVSDVTKAGHSLELIEPLIKFQVGLKKLNLHEEEHVLLMAICIVSPDRPGVQDAALIEAIQDRLSNTLQTYIRCRHPPPGSHLLYAKMIQKLADLRSLNEEHSKQYRCLSFQPECSMKLTPLVLEVFGNEIS',
  'length': 427,
  'molWeight': 48289,
  'crc64': 'F95F300D042C4CB7',
  'md5': '0D963ACD4A34674368324EE026023597'},
 'type': 'UniProtKB reviewed (Swiss-Prot)'}

In [58]:
get_ensembl('ENSMUSG00000041147')


<Response [200]>

In [59]:
get_ensembl('helloworld')


<Response [400]>

In [60]:
get_ensembl('helloworld').json()


{'error': "ID 'helloworld' not found"}

In [61]:
ensembl_parse_response(get_ensembl('ENSMUSG00000041147'))


{'object_type': 'Gene',
 'species': 'mus_musculus',
 'assembly_name': 'GRCm39',
 'biotype': 'protein_coding',
 'display_name': 'Brca2',
 'id': 'ENSMUSG00000041147',
 'db_type': 'core',
 'description': 'breast cancer 2, early onset [Source:MGI Symbol;Acc:MGI:109337]',
 'source': 'ensembl_havana',
 'canonical_transcript': 'ENSMUST00000044620.11'}

In [62]:
main(['P11473', 'Q91XI3', 'hello', 'ENSG00000157764', 'ENSG00000139618'])


,organism,geneInfo,sequenceInfo,type,error,object_type,species,assembly_name,biotype,display_name,id,db_type,description,source,canonical_transcript
P11473,Homo sapiens,[{'geneName': {'evidences': [{'evidenceCode': ...,{'value': 'MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFH...,UniProtKB reviewed (Swiss-Prot),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Q91XI3,Ictidomys tridecemlineatus,[{'geneName': {'value': 'INS'}}],{'value': 'MALWTRLLPLLALLALLGPDPAQAFVNQHLCGSHL...,UniProtKB reviewed (Swiss-Prot),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
hello,NaN,NaN,NaN,NaN,unknown database,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ENSG00000157764,NaN,NaN,NaN,NaN,NaN,Gene,homo_sapiens,GRCh38,protein_coding,BRAF,ENSG00000157764,core,"B-Raf proto-oncogene, serine/threonine kinase ...",ensembl_havana,ENST00000646891.2
ENSG00000139618,NaN,NaN,NaN,NaN,NaN,Gene,homo_sapiens,GRCh38,protein_coding,BRCA2,ENSG00000139618,core,BRCA2 DNA repair associated [Source:HGNC Symbo...,ensembl_havana,ENST00000380152.8
